In [7]:
import base64
import email
from email.header import decode_header
import re
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import pandas as pd

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def clean_header_string(header_value):
    if not header_value:
        return ""
    decoded_fragments = decode_header(header_value)
    pieces = []
    for piece, encoding in decoded_fragments:
        if isinstance(piece, bytes):
            if encoding:
                pieces.append(piece.decode(encoding, errors='ignore'))
            else:
                pieces.append(piece.decode('utf-8', errors='ignore'))
        else:
            pieces.append(str(piece))
    return "".join(pieces)

def get_email_body(mime_msg):
    body = ""
    if mime_msg.is_multipart():
        for part in mime_msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))
            if content_type == "text/plain" and "attachment" not in content_disposition:
                payload = part.get_payload(decode=True)
                if payload:
                    body += payload.decode(part.get_content_charset() or 'utf-8', errors='ignore')
                    break
    else:
        payload = mime_msg.get_payload(decode=True)
        if payload:
            body = payload.decode(mime_msg.get_content_charset() or 'utf-8', errors='ignore')
    return re.sub(r'\s+', ' ', body).strip()

# ==========================================
# 改善版1: 株式会社などを徹底的に除去して略称化
# ==========================================
def clean_from_name(from_user):
    # 1. メールアドレス部分（<...>）を取り除く
    name = re.sub(r'<.*?>', '', from_user).strip()
    name = name.replace('"', '').replace("'", "")
    
    # 2. 法人関連の表記を消去（（株）や [株]、株式会社 など）
    corporate_words = [
        r'株式会社', r'有限会社', r'合同会社', r'一般社団法人', r'公益財団法人',
        r'\(株\)', r'（株）', r'\[株\]', r'【株】',
        r'\(有\)', r'（有）', r'\(同\)', r'（同）'
    ]
    for word in corporate_words:
        name = re.sub(word, '', name)
        
    # 3. ドメイン（.co.jpなど）や余計な役割名をカット
    name = re.sub(r'\.(co\.jp|com|net|org|info|jp)', '', name, flags=re.IGNORECASE)
    extra_words = ['公式', '事務局', 'カスタマーサポート', 'サポート', '窓口', 'オンラインショップ', 'メルマガ編集部']
    for word in extra_words:
        name = name.replace(word, '')
        
    return name.strip() if name.strip() else "不明な送信元"

# ==========================================
# 改善版2: ロジックを細分化したカテゴリ分類
# ==========================================
def classify_subject(subject, has_unsubscribe):
    subj = subject.lower()
    
    # 1. 請求・インボイス（最重要・お金関連）
    if any(k in subj for k in ['請求書', '見積書', 'インボイス', '確定申告', 'お支払い金額', '引き落とし']):
        return '請求・インボイス'
        
    # 2. 注文・購入明細（取引実績）
    if any(k in subj for k in ['注文', '購入', '発送', '決済', '領収書', '商品お届け']):
        return '注文・購入明細'
        
    # 3. 予約・イベント関連
    if any(k in subj for k in ['予約', 'チケット', 'セミナー', 'イベント', 'リマインド', 'お申込み完了']):
        return '予約・イベント'
        
    # 4. 重要なお知らせ（サービス側の規約変更や障害など）
    if any(k in subj for k in ['重要', '規約', '改定', '不具合', '障害', 'メンテナンス', 'お詫び']):
        return '重要なお知らせ・規約変更'

    # 5. システム通知・セキュリティ（自動送信）
    if any(k in subj for k in ['ログイン', 'パスワード', '認証', 'アラート', '検知', '登録完了', '退会完了', '設定変更']):
        return 'システム通知・ログイン'

    # 6. ビジネス連絡・商談
    if any(k in subj for k in ['面談', 'ミーティング', '打ち合わせ', 'mtg', 'お世話になっております', 'ご相談']):
        return 'ビジネス連絡・商談'

    # 7. 個人・SNS・返信（人からの連絡）
    if any(k in subj for k in ['re:', 'fwd:', 'メッセージが届いています', '承認欲求', 'があなたをフォローしました']):
        return '個人・SNS・返信'

    # 8. 会員宛て定期便・マイページ通知（メルマガ一歩手前）
    if any(k in subj for k in ['マイページ', 'レポート', 'ダイジェスト', '新着情報', 'アップデート']):
        return 'マイページ・会員限定'

    # 9. メルマガ・広告・プロモーション（不要メールの主犯格）
    if has_unsubscribe == 1 or any(k in subj for k in ['ニュース', 'レター', 'マガジン', '通信', '【pr】', 'お得', '限定', 'セール', 'キャンペーン', '割引', 'クーポン']):
        return 'メルマガ・プロモーション'

    # 10. どれにも該当しない場合
    return 'その他一般受信'

# ==========================================
# メイン処理（前回同様）
# ==========================================
def get_gmail_messages(max_results=100):
    flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
    creds = flow.run_local_server(port=0)
    service = build('gmail', 'v1', credentials=creds)
    
    results = service.users().messages().list(userId='me', maxResults=max_results, labelIds=['INBOX']).execute()
    messages = results.get('messages', [])
    
    if not messages:
        print('メールが見つかりませんでした。')
        return

    print(f"\n--- メールの詳細データを {len(messages)} 件取得・高度分類中... ---")
    
    mail_data_list = []
    
    for idx, msg in enumerate(messages):
        try:
            txt = service.users().messages().get(userId='me', id=msg['id'], format='raw').execute()
            msg_bytes = base64.urlsafe_b64decode(txt['raw'].encode('ASCII'))
            mime_msg = email.message_from_bytes(msg_bytes)
            
            subject = clean_header_string(mime_msg.get('Subject', '無題'))
            from_user = clean_header_string(mime_msg.get('From', '不明'))
            date_str = mime_msg.get('Date', '')
            
            email_address_match = re.search(r'[\w\.-]+@[\w\.-]+', from_user)
            from_email = email_address_match.group(0) if email_address_match else from_user
            has_unsubscribe = 1 if mime_msg.get('List-Unsubscribe') else 0
            
            # 強化したクレンジング＆分類を適用
            short_from_name = clean_from_name(from_user)
            category = classify_subject(subject, has_unsubscribe)
            
            body_text = get_email_body(mime_msg)
            
            mail_data_list.append({
                'Message-ID': msg['id'],
                'Date': date_str,
                'From-Name': short_from_name,
                'From-Email': from_email,
                'Subject': subject,
                'Category': category,
                'Has-Unsubscribe': has_unsubscribe,
                'Snippet': body_text[:100]
            })
            
            if (idx + 1) % 20 == 0:
                print(f"{idx + 1} 件完了...")
                
        except Exception as e:
            print(f"エラー（ID: {msg['id']}）: {e}")
            continue

    df = pd.DataFrame(mail_data_list)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    
    output_filename = 'gmail_analysis_data.csv'
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\nデータのエクスポートが完了しました: {output_filename}")

if __name__ == '__main__':
    # しっかり分析するために、まずは100件〜200件ほどで試すのがおすすめです
    get_gmail_messages(max_results=100)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=932046849021-ctbda1i4agn6ff6ccm0ovtcqmaueko39.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A53872%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=e92KhB9t34gJHZX0HFyM6JTfIC6f6L&code_challenge=sy07iwMh037r4pTsNkjRebbtRDaNqdO0hgHWFNAxEiQ&code_challenge_method=S256&access_type=offline

--- メールの詳細データを 100 件取得・高度分類中... ---
20 件完了...
40 件完了...
60 件完了...
80 件完了...
100 件完了...

データのエクスポートが完了しました: gmail_analysis_data.csv


In [8]:
df = pd.read_csv('gmail_analysis_data.csv')
display(df.head())

,Message-ID,Date,From-Name,From-Email,Subject,Category,Has-Unsubscribe,Snippet
0,19e82439775eadd9,2026-06-01 08:18:46+00:00,Google,no-reply@accounts.google.com,セキュリティ通知,その他一般受信,0,[image: Google] あなたは GmailAnalyzer に Google アカ...
1,19e81970a5149dae,NaN,Amazon,shipment-tracking@amazon.co.jp,発送済み：「ドラえもんの理科おもしろ攻略7冊セット...」,注文・購入明細,0,注文履歴 https://www.amazon.co.jp/gp/css/order-his...
2,19e8162836314a7a,NaN,楽天カード,info@mail.rakuten-card.co.jp,【速報版】カード利用のお知らせ(本人ご利用分),その他一般受信,0,━━━━━━━━━━ 【速報版】カード利用お知らせメール ━━━━━━━━━━ 楽天カード（...
3,19e815dcbf329356,NaN,東京都水道局,info@tokyo.suidoapp.jp,「動画制作ラボ」～山之内すずと動画を作ろう！～,その他一般受信,0,【東京都水道局からのお知らせ】 「動画制作ラボ」～山之内すずと動画を作ろう～ 東京都水道局で...
4,19e8159cd6d8de33,NaN,R-AGENT求人紹介,customer-center213@r-agent.com,□OMO企画推進担当 ◎オン/オフラインを融合した新しい顧客体験の戦略設計などのおすすめ求人...,その他一般受信,0,"<!DOCTYPE HTML PUBLIC ""-//W3C//DTD HTML 4.01 T..."


In [ ]:
df['date']

Message-ID         object
Date               object
From-Name          object
From-Email         object
Subject            object
Category           object
Has-Unsubscribe     int64
Snippet            object
dtype: object